In [2]:
import pandas as pd


mitocarta = pd.read_excel("../data/raw/Human.MitoCarta3.0.xls",sheet_name="A Human MitoCarta3.0").drop(["HumanGeneID","MouseOrthologGeneID"],axis=1).set_index("Symbol")


complexes = {
    "Complex I": "Complex I",
    "Complex II": "Complex II",
    "Complex III": "Complex III",
    "Complex IV": "Complex IV",
    "Complex V": "Complex V",
}

oxphos_genes = {}

for name, pattern in complexes.items():

    subset = mitocarta[
        mitocarta["MitoCarta3.0_MitoPathways"]
        .str.contains(pattern, case=False, na=False)
    ]

    genes = set()

    for symbol, row in subset.iterrows():

        # Official symbol
        genes.add(symbol.upper())

        # Synonyms
        if pd.notna(row["Synonyms"]):
            genes.update(
                syn.strip().upper()
                for syn in row["Synonyms"].split("|")
                if syn.strip()
            )

    oxphos_genes[name] = genes
    
for complex_name, genes in oxphos_genes.items():
    print(f"{complex_name}: {len(genes)} identifiers")
    print(sorted(list(genes))[:10], "\n")
    

from pathlib import Path

subtypes = ["classical", "proneural", "mesenchymal"]

input_dir = Path("../data/processed/new_diff")
output_dir = Path("../data/processed/new_oxphos")
output_dir.mkdir(exist_ok=True)

for subtype in subtypes:

    df = pd.read_csv(
        input_dir / f"CPTAC_{subtype}_vs_GTEx7_PyDESeq2_results.csv",
        index_col=0
    )

    print(df.index)
    # Ensure uppercase gene names
    df.index = df.index.str.upper()

    for complex_name, genes in oxphos_genes.items():

        subset = df.loc[
            df.index.intersection(genes)
        ].sort_index()

        filename = (
            f"{subtype}_"
            f"{complex_name.lower().replace(' ', '_')}.csv"
        )

        subset.to_csv(output_dir / filename)

        print(f"{filename}: {subset.shape[0]} genes")

Complex I: 561 identifiers
['-', '0710008D09RIK', '2010204O13RIK', '2310061C15RIK', '2P1', '6330578E17RIK', 'ACAD9', 'ACN9', 'ACP', 'ACP1'] 

Complex II: 124 identifiers
['0710008D09RIK', '2010204O13RIK', 'ACN9', 'BA6B20.2', 'BCS', 'BCS1', 'BCS1L', 'BFZB', 'BJS', 'C11ORF79'] 

Complex III: 82 identifiers
['0710008D09RIK', '2010204O13RIK', 'BA6B20.2', 'BCS', 'BCS1', 'BCS1L', 'BFZB', 'BJS', 'C11ORF83', 'C20ORF44'] 

Complex IV: 179 identifiers
['-', '2310061C15RIK', '6330578E17RIK', 'APLCC', 'APOP', 'APOP1', 'APOPT1', 'BRP17', 'C12ORF62', 'C14ORF112'] 

Complex V: 107 identifiers
['6.8PL', 'APT5H', 'ATP11', 'ATP11P', 'ATP12', 'ATP12P', 'ATP5', 'ATP5A', 'ATP5A1', 'ATP5AL2'] 

Index(['NBEAL1', 'NGRN', 'BSCL2', 'BCAP29', 'ATP6V0C', 'VPS33B', 'PPP3R1',
       'SUPT4H1', 'ARL6IP4', 'RPS10',
       ...
       'AL583810.1', 'AL355075.2', 'APC2', 'AC093536.1', 'B3GALNT2P1',
       'AL139353.1', 'AC090616.2', 'ADPRS', 'N4BP1', 'CABP5'],
      dtype='object', name='gene_name', length=45019)
classi